In [1]:
from pathlib import Path
import json
import math

import torch
import torchaudio


PROJECT_ROOT = Path("/workspace/storyfx")

# --------------------------------------------------
# Mixing parameters
# --------------------------------------------------

AMBIENT_RELATIVE_DB = -10.0

DISCRETE_FADE_MS = 80
AMBIENT_EDGE_FADE_MS = 500
AMBIENT_LOOP_CROSSFADE_MS = 500

LOCAL_RMS_WINDOW_S = 2.0

EPS = 1e-8


In [2]:
def load_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def extract_excerpt_id(path, suffix):
    path = Path(path)
    filename = path.stem

    if filename.startswith("pred_"):
        filename = filename[len("pred_"):]

    if not filename.endswith(suffix):
        raise ValueError(
            f"Expected filename ending with '{suffix}', "
            f"got: {path.name}"
        )

    return filename[:-len(suffix)]


In [3]:
def load_audio(path):
    waveform, sample_rate = torchaudio.load(
        str(path)
    )

    return waveform.float(), sample_rate


def save_audio(path, waveform, sample_rate):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    torchaudio.save(
        str(path),
        waveform.cpu(),
        sample_rate,
    )


In [4]:
def match_channels(audio, target_channels):
    channels = audio.shape[0]

    if channels == target_channels:
        return audio

    if channels == 1 and target_channels == 2:
        return audio.repeat(2, 1)

    if channels == 2 and target_channels == 1:
        return audio.mean(
            dim=0,
            keepdim=True,
        )

    raise ValueError(
        f"Unsupported channel conversion: "
        f"{channels} -> {target_channels}"
    )


def resample_if_needed(
    audio,
    source_sr,
    target_sr,
):
    if source_sr == target_sr:
        return audio

    print(
        f"[RESAMPLE] "
        f"{source_sr} Hz -> {target_sr} Hz"
    )

    return torchaudio.functional.resample(
        audio,
        source_sr,
        target_sr,
    )


In [5]:
def rms(audio):
    if audio.numel() == 0:
        return 0.0

    return float(
        torch.sqrt(
            torch.mean(audio ** 2) + EPS
        )
    )


def db_to_linear(db):
    return 10.0 ** (db / 20.0)


In [6]:
def level_match_to_voice(
    sfx,
    voice_reference,
):
    sfx_rms = rms(sfx)
    voice_rms = rms(voice_reference)

    if sfx_rms <= EPS:
        return sfx

    if voice_rms <= EPS:
        return sfx

    gain = (
        voice_rms
        / sfx_rms
    )

    return sfx * gain


def level_match_relative_to_voice(
    sfx,
    voice_reference,
    relative_db,
):
    sfx_rms = rms(sfx)
    voice_rms = rms(voice_reference)

    if sfx_rms <= EPS:
        return sfx

    if voice_rms <= EPS:
        return sfx

    target_rms = (
        voice_rms
        * db_to_linear(relative_db)
    )

    gain = (
        target_rms
        / sfx_rms
    )

    return sfx * gain


In [7]:
def apply_edge_fades(
    audio,
    sample_rate,
    fade_ms,
):
    audio = audio.clone()

    fade_samples = int(
        sample_rate
        * fade_ms
        / 1000
    )

    fade_samples = min(
        fade_samples,
        audio.shape[-1] // 2,
    )

    if fade_samples <= 0:
        return audio

    fade_in = torch.linspace(
        0.0,
        1.0,
        fade_samples,
        device=audio.device,
    )

    fade_out = torch.linspace(
        1.0,
        0.0,
        fade_samples,
        device=audio.device,
    )

    audio[:, :fade_samples] *= fade_in
    audio[:, -fade_samples:] *= fade_out

    return audio


In [8]:
def get_reference_window(
    narration,
    sample_rate,
    center_time,
    window_s=2.0,
):
    half_window = window_s / 2.0

    start_time = max(
        0.0,
        center_time - half_window,
    )

    end_time = min(
        narration.shape[-1] / sample_rate,
        center_time + half_window,
    )

    start_sample = int(
        start_time * sample_rate
    )

    end_sample = int(
        end_time * sample_rate
    )

    return narration[
        :,
        start_sample:end_sample,
    ]


In [9]:
def mix_at_time(
    base,
    overlay,
    sample_rate,
    start_time,
):
    start_sample = int(
        round(start_time * sample_rate)
    )

    if start_sample >= base.shape[-1]:
        return base

    end_sample = min(
        start_sample + overlay.shape[-1],
        base.shape[-1],
    )

    overlay_length = (
        end_sample - start_sample
    )

    base[
        :,
        start_sample:end_sample,
    ] += overlay[
        :,
        :overlay_length,
    ]

    return base


In [10]:
def loop_with_crossfade(
    audio,
    target_samples,
    sample_rate,
    crossfade_ms=500,
):
    if target_samples <= 0:
        return audio[:, :0]

    if audio.shape[-1] >= target_samples:
        return audio[:, :target_samples].clone()

    crossfade_samples = int(
        sample_rate
        * crossfade_ms
        / 1000
    )

    crossfade_samples = min(
        crossfade_samples,
        audio.shape[-1] // 2,
    )

    if crossfade_samples <= 0:
        repeats = math.ceil(
            target_samples
            / audio.shape[-1]
        )

        return audio.repeat(
            1,
            repeats,
        )[:, :target_samples]

    output = audio.clone()

    fade_out = torch.linspace(
        1.0,
        0.0,
        crossfade_samples,
        device=audio.device,
    )

    fade_in = torch.linspace(
        0.0,
        1.0,
        crossfade_samples,
        device=audio.device,
    )

    while output.shape[-1] < target_samples:
        overlap = min(
            crossfade_samples,
            output.shape[-1],
            audio.shape[-1],
        )

        left = output[
            :,
            -overlap:,
        ]

        right = audio[
            :,
            :overlap,
        ]

        blended = (
            left
            * fade_out[-overlap:]
            +
            right
            * fade_in[-overlap:]
        )

        output = torch.cat(
            [
                output[:, :-overlap],
                blended,
                audio[:, overlap:],
            ],
            dim=-1,
        )

    return output[
        :,
        :target_samples,
    ]


In [11]:
def discrete_sfx_mixing(
    narration,
    sample_rate,
    schedule,
):
    schedule = Path(schedule)

    discrete_schedule = load_json(
        schedule
    )

    excerpt_id = extract_excerpt_id(
        schedule,
        "_discrete_sfx_schedule",
    )

    wet_dir = (
        PROJECT_ROOT
        / "outputs"
        / "wet_mixes"
        / excerpt_id
    )

    mixed = narration.clone()

    print(
        f"Mixing discrete SFX for {excerpt_id}"
    )

    for sfx in discrete_schedule:
        sfx_id = sfx[
            "discrete_sfx_id"
        ]

        anchor_time = float(
            sfx["anchor_time"]
        )

        asset_path = (
            wet_dir
            / f"{sfx_id}.wav"
        )

        if not asset_path.exists():
            raise FileNotFoundError(
                f"Discrete wet mix not found: "
                f"{asset_path}"
            )

        asset, asset_sr = load_audio(
            asset_path
        )

        asset = resample_if_needed(
            asset,
            asset_sr,
            sample_rate,
        )

        asset = match_channels(
            asset,
            narration.shape[0],
        )

        reference = get_reference_window(
            narration,
            sample_rate,
            anchor_time,
            LOCAL_RMS_WINDOW_S,
        )

        before_rms = rms(asset)
        voice_rms = rms(reference)

        asset = level_match_to_voice(
            asset,
            reference,
        )

        after_rms = rms(asset)

        asset = apply_edge_fades(
            asset,
            sample_rate,
            DISCRETE_FADE_MS,
        )

        mixed = mix_at_time(
            mixed,
            asset,
            sample_rate,
            anchor_time,
        )

        print(
            f"[DISCRETE] {sfx_id} "
            f"@ {anchor_time:.2f}s | "
            f"SFX RMS {before_rms:.5f} -> {after_rms:.5f} | "
            f"voice RMS {voice_rms:.5f}"
        )

    return mixed


In [12]:
def ambient_sfx_mixing(
    mix,
    narration,
    sample_rate,
    segment_schedule,
):
    segment_schedule = Path(
        segment_schedule
    )

    segments = load_json(
        segment_schedule
    )

    excerpt_id = extract_excerpt_id(
        segment_schedule,
        "_segment_schedule",
    )

    ambient_dir = (
        PROJECT_ROOT
        / "outputs"
        / "assets"
        / excerpt_id
        / "ambient_assets"
    )

    mixed = mix.clone()

    if not ambient_dir.exists():
        print(
            f"No ambient asset directory found: "
            f"{ambient_dir}"
        )
        return mixed

    print(
        f"Mixing ambient SFX for {excerpt_id}"
    )

    layers = (
        "place",
        "weather",
        "special",
    )

    for segment in segments:
        segment_id = segment[
            "segment_id"
        ]

        start_time = float(
            segment["start_time"]
        )

        end_time = float(
            segment["end_time"]
        )

        start_sample = int(
            round(
                start_time
                * sample_rate
            )
        )

        end_sample = int(
            round(
                end_time
                * sample_rate
            )
        )

        end_sample = min(
            end_sample,
            narration.shape[-1],
        )

        target_samples = (
            end_sample
            - start_sample
        )

        if target_samples <= 0:
            continue

        narration_reference = narration[
            :,
            start_sample:end_sample,
        ]

        for layer in layers:
            asset_path = (
                ambient_dir
                / f"{segment_id}_{layer}.wav"
            )

            if not asset_path.exists():
                continue

            asset, asset_sr = load_audio(
                asset_path
            )

            asset = resample_if_needed(
                asset,
                asset_sr,
                sample_rate,
            )

            asset = match_channels(
                asset,
                narration.shape[0],
            )

            looped = loop_with_crossfade(
                asset,
                target_samples,
                sample_rate,
                AMBIENT_LOOP_CROSSFADE_MS,
            )

            before_rms = rms(looped)
            voice_rms = rms(
                narration_reference
            )

            looped = level_match_relative_to_voice(
                looped,
                narration_reference,
                AMBIENT_RELATIVE_DB,
            )

            after_rms = rms(looped)

            looped = apply_edge_fades(
                looped,
                sample_rate,
                AMBIENT_EDGE_FADE_MS,
            )

            mixed[
                :,
                start_sample:end_sample,
            ] += looped[
                :,
                :target_samples,
            ]

            print(
                f"[AMBIENT] "
                f"{segment_id}_{layer} "
                f"{start_time:.2f}s -> {end_time:.2f}s | "
                f"SFX RMS {before_rms:.5f} -> {after_rms:.5f} | "
                f"voice RMS {voice_rms:.5f} | "
                f"{AMBIENT_RELATIVE_DB:+.1f} dB"
            )

    return mixed


In [13]:
EXCERPT_ID = "excerpt_001"

NARRATION_PATH = (
    PROJECT_ROOT
    / "data"
    / EXCERPT_ID
    / "raw"
    / f"{EXCERPT_ID}.wav"
)

DISCRETE_SCHEDULE_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "schedules"
    / f"{EXCERPT_ID}_discrete_sfx_schedule.json"
)

SEGMENT_SCHEDULE_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "schedules"
    / f"{EXCERPT_ID}_segment_schedule.json"
)

FINAL_OUTPUT_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "augumented_stories"
    / f"{EXCERPT_ID}_augumented.wav"
)


narration, sample_rate = load_audio(
    NARRATION_PATH
)

print(
    f"Narration: {NARRATION_PATH}"
)

print(
    f"Sample rate: {sample_rate} Hz"
)

print(
    f"Duration: "
    f"{narration.shape[-1] / sample_rate:.2f}s"
)


/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

Narration: /workspace/storyfx/data/excerpt_001/raw/excerpt_001.wav
Sample rate: 44100 Hz
Duration: 388.60s


In [14]:
mix = discrete_sfx_mixing(
    narration,
    sample_rate,
    DISCRETE_SCHEDULE_PATH,
)

mix = ambient_sfx_mixing(
    mix,
    narration,
    sample_rate,
    SEGMENT_SCHEDULE_PATH,
)

peak = float(
    torch.max(
        torch.abs(mix)
    )
)

print(
    f"\nFinal peak amplitude: {peak:.3f}"
)

if peak > 1.0:
    print(
        "[WARNING] Final mix exceeds 1.0. "
        "No normalization was applied."
    )

save_audio(
    FINAL_OUTPUT_PATH,
    mix,
    sample_rate,
)

print(
    "\nFinal augmented story saved to:"
)

print(
    FINAL_OUTPUT_PATH
)


Mixing discrete SFX for excerpt_001
[RESAMPLE] 48000 Hz -> 44100 Hz
[DISCRETE] discrete_sfx000 @ 95.24s | SFX RMS 0.04786 -> 0.09707 | voice RMS 0.09707
[RESAMPLE] 48000 Hz -> 44100 Hz
[DISCRETE] discrete_sfx001 @ 104.19s | SFX RMS 0.00511 -> 0.08690 | voice RMS 0.08692
[RESAMPLE] 48000 Hz -> 44100 Hz
[DISCRETE] discrete_sfx002 @ 107.11s | SFX RMS 0.09790 -> 0.07932 | voice RMS 0.07932
[RESAMPLE] 48000 Hz -> 44100 Hz
[DISCRETE] discrete_sfx003 @ 112.36s | SFX RMS 0.05177 -> 0.10131 | voice RMS 0.10131
[RESAMPLE] 48000 Hz -> 44100 Hz
[DISCRETE] discrete_sfx004 @ 163.15s | SFX RMS 0.26361 -> 0.09305 | voice RMS 0.09305
[RESAMPLE] 48000 Hz -> 44100 Hz
[DISCRETE] discrete_sfx005 @ 172.27s | SFX RMS 0.19532 -> 0.10631 | voice RMS 0.10631
[RESAMPLE] 48000 Hz -> 44100 Hz
[DISCRETE] discrete_sfx006 @ 218.66s | SFX RMS 0.01298 -> 0.07261 | voice RMS 0.07261
[RESAMPLE] 48000 Hz -> 44100 Hz
[DISCRETE] discrete_sfx007 @ 307.79s | SFX RMS 0.00660 -> 0.07049 | voice RMS 0.07050
[RESAMPLE] 48000 Hz -

/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:337: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.save_with_torchcodec` under the hood. Some parameters like format, encoding, bits_per_sample, buffer_size, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's encoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.encoders.AudioEncoder
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:247: UserWarning: torio.io._streaming_media_encoder.StreamingMediaEncoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  s 


Final augmented story saved to:
/workspace/storyfx/outputs/augumented_stories/excerpt_001_augumented.wav
